# Smart Batching Search - Testing Notebook

This notebook demonstrates and tests the smart batching search functionality.

## Features
- **Planning**: Organize search using smart batching
- **Execution**: Execute search with proportional sampling
- **Rate Limiting**: Configurable requests per minute
- **Parallel Processing**: Efficient parallel execution

## Configuration

**Environment Variables:** This notebook loads configuration from a `.env` file in the `Smart_Batching` directory.

Create a `.env` file with:
```
BIGDATA_API_KEY=your_api_key_here
BIGDATA_API_BASE_URL=https://api.bigdata.com
```

**Options for API_BASE_URL:**
- `https://api.bigdata.com` (production - default)

**Note:** You must restart the kernel and run cells from the beginning if you change the API Base URL, as it's read at import time.

## 1. Load Environment Variables and Setup

**IMPORTANT:** Load `.env` file and set API_BASE_URL here before importing modules, as it's read at import time.

In [1]:
# Library imports
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
env_path = Path.cwd() / ".env"
if env_path.exists():
    load_dotenv(env_path)

# Set API base URL BEFORE importing (smart_batching_config reads it at import time)
API_BASE_URL = os.getenv("BIGDATA_API_BASE_URL", "https://api.bigdata.com")
os.environ["BIGDATA_API_BASE_URL"] = API_BASE_URL

# Add current directory to path for local imports
sys.path.insert(0, str(Path.cwd()))

# Import all from src module
from src import (
    plan_search,
    execute_search,
    deduplicate_documents,
    save_plan,
    load_plan,
    load_universe_from_csv,
    convert_to_dataframe,
)

## 2. Configuration

In [2]:
# Configuration
# Note: API_BASE_URL and API_KEY are loaded from .env file in cell 2
# To change them, edit the .env file or set environment variables:
#   export BIGDATA_API_KEY='your_api_key_here'
#   export BIGDATA_API_BASE_URL='https://api.bigdata.com'

# API Key (loaded from .env in cell 2)
API_KEY = os.getenv("BIGDATA_API_KEY")

if not API_KEY:
    print("⚠️  BIGDATA_API_KEY not set. Please set it in your .env file:")
    print("   BIGDATA_API_KEY=your_api_key_here")
    print("   Or set environment variable: export BIGDATA_API_KEY='your_api_key_here'")
else:
    print(f"✅ API Key configured: {API_KEY[:8]}...{API_KEY[-4:]}")

# Test parameters
TEST_TEXT = "Decline in customer confidence in the company"
TEST_UNIVERSE_CSV = "id_name_mapping_us_top_3000.csv"
# TEST_UNIVERSE_CSV = "sample_universe.csv"  # small test universe
TEST_START_DATE = "2021-01-01"
TEST_END_DATE = "2021-06-30"
TEST_CHUNK_PERCENTAGE = 0.1  # 10% of total chunks

print(f"\n📝 Test Configuration:")
print(f"   API Base URL: {API_BASE_URL}")
print(f"   Text: '{TEST_TEXT}'")
print(f"   Universe: {TEST_UNIVERSE_CSV}")
print(f"   Date Range: {TEST_START_DATE} to {TEST_END_DATE}")
print(f"   Chunk Percentage: {TEST_CHUNK_PERCENTAGE*100:.0f}%")

✅ API Key configured: bd_v1_NW...2252

📝 Test Configuration:
   API Base URL: https://api.bigdata.com
   Text: 'Decline in customer confidence in the company'
   Universe: id_name_mapping_us_top_3000.csv
   Date Range: 2021-01-01 to 2021-06-30
   Chunk Percentage: 10%


## 3. Test Universe Loading

In [3]:
# Test loading universe from CSV
try:
    companies = load_universe_from_csv(TEST_UNIVERSE_CSV)
    print(f"✅ Loaded {len(companies)} companies from {TEST_UNIVERSE_CSV}")
    print(f"   First 5 companies: {companies[:5]}")
except Exception as e:
    print(f"❌ Error loading universe: {e}")

2026-01-30 16:37:42,736 - INFO - Loaded 4731 entity IDs from id_name_mapping_us_top_3000.csv
✅ Loaded 4731 companies from id_name_mapping_us_top_3000.csv
   First 5 companies: ['00067A', '001F1B', '002A99', '00326D', '003B70']


## 4. Step 1: Plan Search

In [4]:
# Plan the search
if API_KEY:
    print("📋 Planning search...")
    print("-" * 80)
    
    try:
        plan = plan_search(
            text=TEST_TEXT,
            universe_csv_path=TEST_UNIVERSE_CSV,
            start_date=TEST_START_DATE,
            end_date=TEST_END_DATE,
            api_key=API_KEY,
            api_base_url=API_BASE_URL,
            volume_query_mode="iterative",
            max_iterations_per_batch=10
        )
        
        print(f"\n✅ Planning complete!")
        print(f"   Total expected chunks: {plan['total_expected_chunks']:,}")
        print(f"   Number of baskets: {len(plan['baskets'])}")
        
        if plan.get('planning_metadata'):
            metadata = plan['planning_metadata']
            print(f"   Total companies: {metadata.get('total_companies', 'N/A')}")
            print(f"   Companies with chunks: {metadata.get('companies_with_chunks', 'N/A')}")
            print(f"   Uses smart batching: {metadata.get('uses_smart_batching', False)}")
        
        # Show example basket
        if plan['baskets']:
            example_basket = plan['baskets'][0]
            print(f"\n   Example Basket:")
            print(f"     Basket ID: {example_basket['basket_id']}")
            print(f"     Expected chunks: {example_basket['expected_chunks']}")
            print(f"     Companies: {len(example_basket['companies'])} companies")
            print(f"     Query text: '{example_basket['query']['text']}'")
            print(f"     Max chunks in query: {example_basket['query']['max_chunks']}")
        
    except Exception as e:
        print(f"❌ Error during planning: {e}")
        import traceback
        traceback.print_exc()
        plan = None
else:
    print("⚠️  Skipping planning - API key not set")
    plan = None

📋 Planning search...
--------------------------------------------------------------------------------
2026-01-30 16:37:42,741 - INFO - Planning search for text: 'Decline in customer confidence in the company'
2026-01-30 16:37:42,741 - INFO - Date range: 2021-01-01 to 2021-06-30
2026-01-30 16:37:42,742 - INFO - Loaded 4731 entity IDs from id_name_mapping_us_top_3000.csv
2026-01-30 16:37:42,742 - INFO - Loaded 4731 companies from universe
2026-01-30 16:37:42,747 - INFO - Using SmartBatchingPlanner with adaptive time-window splitting
PHASE 1: Querying full period for all companies (2021-01-01 to 2021-06-30)
         Mode: iterative
    [ITERATIVE MODE] Querying 4731 companies in 10 batches of 500
    Each batch iterates until no new companies are found (max 10 iterations)
      Batch 1/10, Iter 1: Found 80 new companies, 420 remaining
      Batch 1/10, Iter 2: Found 128 new companies, 292 remaining
      Batch 1/10, Iter 3: Found 120 new companies, 172 remaining
      Batch 1/10, Iter 4: 

## 5. Save Plan (Optional)

In [5]:
# Save plan for later use
if plan:
    plan_file = "test_search_plan.json"
    try:
        save_plan(plan, plan_file)
        print(f"✅ Plan saved to {plan_file}")
        print(f"   You can load it later with: plan = load_plan('{plan_file}')")
    except Exception as e:
        print(f"❌ Error saving plan: {e}")

2026-01-30 16:41:15,425 - INFO - Plan saved to test_search_plan.json
✅ Plan saved to test_search_plan.json
   You can load it later with: plan = load_plan('test_search_plan.json')


## 6. Step 2: Execute Search with Proportional Sampling

In [6]:
# Execute search with proportional sampling
if plan and API_KEY:
    print("🔍 Executing search...")
    print("-" * 80)
    
    try:
        results_raw = execute_search(
            search_plan=plan,
            chunk_percentage=0.1,
            requests_per_minute=100,  # Rate limit
            api_key=API_KEY,
            api_base_url=API_BASE_URL,
        )
        
        results = deduplicate_documents(results_raw)

        print(f"\n✅ Search complete!")
        print(f"   Retrieved {len(results):,} deduplicated chunks")
            
    except Exception as e:
        print(f"❌ Error during execution: {e}")
        import traceback
        traceback.print_exc()
        results = []
else:
    print("⚠️  Skipping execution - plan or API key not available")
    results = []

🔍 Executing search...
--------------------------------------------------------------------------------
2026-01-30 16:41:15,429 - INFO - Executing search with 10.0% of chunks
2026-01-30 16:41:15,429 - INFO - Total maximum expected chunks: 27,302
2026-01-30 16:41:15,430 - INFO - Searching 319 baskets
2026-01-30 16:41:16,606 - INFO - Basket high_basket_2: Retrieved 70 documents with 80 chunks
2026-01-30 16:41:16,957 - INFO - Basket high_basket_3: Retrieved 81 documents with 89 chunks
2026-01-30 16:41:17,238 - INFO - Basket high_basket_5: Retrieved 68 documents with 75 chunks
2026-01-30 16:41:17,437 - INFO - Basket high_basket_7: Retrieved 64 documents with 67 chunks
2026-01-30 16:41:17,467 - INFO - Basket high_basket_6: Retrieved 66 documents with 82 chunks
2026-01-30 16:41:17,494 - INFO - Basket high_basket_1: Retrieved 67 documents with 94 chunks
2026-01-30 16:41:17,827 - INFO - Basket high_basket_4: Retrieved 62 documents with 84 chunks
2026-01-30 16:41:18,148 - INFO - Basket high_bask

## 7. Analyze Results

In [7]:
# Convert to DataFrame (exploded by chunk)
df = convert_to_dataframe(results)
df.head()

,date,doc_id,headline,source_id,source_name,source_rank,chunk_index,chunk_text,chunk_relevance,chunk_sentiment,entity_ids,url,reporting_entities
0,2021-03-24,23AF15FF849A7EDB068B24A68AFA5FED,"GameStop loses luster in earnings aftermath, p...",208421,Bloomberg News,RANK_1,9,Customer Service Chief Executive Officer Georg...,0.232903,0.49,"[C4F920, AMR2LI, AMR2LI, D42DBA, D42DBA, D42DB...",https://www.bnnbloomberg.ca/gamestop-loses-lus...,[]
1,2021-04-16,F7827CFDD198914F97EE361938681710,'Four Horsemen' Driving The Retail Trading Eup...,5A5702,Benzinga,RANK_1,7,"In particular, she said it's extremely rare fo...",0.214304,-0.11,"[0CCA2E, C12ED9, D42DBA, E735C9, E735C9]",,[]
2,2021-03-23,5AFA8E72E42F4C50B5A438EE77735FE6,"GameStop earnings fall short of expectations, ...",2435A4,CNN,RANK_1,3,That bodes well for the company's effort to tr...,0.178838,0.23,"[DC1A9F, 5AF7E2, 5E647F, 579D51, 09CF77, C7051...",https://www.cnn.com/2021/03/23/tech/gamestop-e...,[]
3,2021-03-23,5AFA8E72E42F4C50B5A438EE77735FE6,"GameStop earnings fall short of expectations, ...",2435A4,CNN,RANK_1,5,Many investors hope Cohen will help make the g...,0.092909,-0.71,"[34A5B5, 103AC2, 42F6D5, 3B6F9E, C4CFC3, D42DB...",https://www.cnn.com/2021/03/23/tech/gamestop-e...,[]
4,2021-03-23,2EA50681CB91057BC8DB95D27848BF3D,GameStop CCO Resigns Ahead Of Q4 Earnings Report,5A5702,Benzinga,RANK_1,2,Why It's Important: GameStop's brick-and-morta...,0.171861,-0.71,"[7D8F56, D42DBA, D42DBA, D42DBA, E735C9, 471AC...",,[]


In [8]:
#Analyze results
if results:

    print("📈 Results Analysis")
    print("-" * 80)
    
    # Summary stats
    n_docs = df['doc_id'].nunique()
    n_chunks = len(df)
    print(f"\n   Total: {n_docs:,} documents, {n_chunks:,} chunks")
    
    # Relevance distribution
    if 'chunk_relevance' in df.columns and df['chunk_relevance'].notna().any():
        print(f"\n   Relevance Scores:")
        print(f"     Min: {df['chunk_relevance'].min():.3f}")
        print(f"     Max: {df['chunk_relevance'].max():.3f}")
        print(f"     Avg: {df['chunk_relevance'].mean():.3f}")
    
    # Sentiment distribution
    if 'chunk_sentiment' in df.columns and df['chunk_sentiment'].notna().any():
        sentiments = df['chunk_sentiment'].dropna()
        positive = (sentiments > 0).sum()
        negative = (sentiments < 0).sum()
        neutral = len(sentiments) - positive - negative
        print(f"\n   Sentiment Distribution:")
        print(f"     Positive: {positive} ({positive/len(sentiments)*100:.1f}%)")
        print(f"     Negative: {negative} ({negative/len(sentiments)*100:.1f}%)")
        print(f"     Neutral: {neutral} ({neutral/len(sentiments)*100:.1f}%)")
    
    # Source distribution
    if 'source_name' in df.columns:
        source_counts = df.groupby('source_name').size().sort_values(ascending=False)
        print(f"\n   Top Sources:")
        for source, count in source_counts.head(5).items():
            print(f"     {source}: {count} chunks")
    
    # Show DataFrame info
    print(f"\n   DataFrame shape: {df.shape}")
    
    # Save results
    from datetime import datetime
    results_file = f"search_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    df.to_json(results_file, orient='records', indent=2)
    print(f"\n💾 Saved to {results_file}")

else:
    df = None
    print("⚠️  No results to analyze")

📈 Results Analysis
--------------------------------------------------------------------------------

   Total: 14,106 documents, 20,837 chunks

   Relevance Scores:
     Min: 0.011
     Max: 0.482
     Avg: 0.077

   Sentiment Distribution:
     Positive: 11061 (53.1%)
     Negative: 9543 (45.8%)
     Neutral: 231 (1.1%)

   Top Sources:
     Factset Transcripts: 7213 chunks
     Quartr Reports: 3781 chunks
     Benzinga: 2603 chunks
     Quartr Transcripts: 2105 chunks
     MT Newswires: 849 chunks

   DataFrame shape: (20837, 13)

💾 Saved to search_results_20260130_164434.json


## 8. Summary

In [9]:
print("=" * 80)
print("Smart Batching Search - Test Summary")
print("=" * 80)

if plan:
    print(f"✅ Planning: SUCCESS")
    print(f"   Expected chunks: {plan['total_expected_chunks']:,}")
    print(f"   Baskets created: {len(plan['baskets'])}")
else:
    print("⚠️  Planning: Not completed")

if results:
    print(f"✅ Execution: SUCCESS")
    print(f"   Chunks retrieved: {len(results):,}")
    print(f"   Percentage used: {TEST_CHUNK_PERCENTAGE*100:.0f}%")
    if plan:
        expected = plan['total_expected_chunks']
        actual = len(results)
        if expected > 0:
            print(f"   Actual vs Expected: {actual/expected*100:.1f}%")
else:
    print("⚠️  Execution: Not completed")

print("\n" + "=" * 80)
print("Test complete!")
print("=" * 80)

Smart Batching Search - Test Summary
✅ Planning: SUCCESS
   Expected chunks: 273,029
   Baskets created: 319
✅ Execution: SUCCESS
   Chunks retrieved: 14,106
   Percentage used: 10%
   Actual vs Expected: 5.2%

Test complete!


## 9. Load Saved Plan (Optional)

In [10]:
# Load a previously saved plan
plan_file = "test_search_plan.json"

if os.path.exists(plan_file):
    try:
        loaded_plan = load_plan(plan_file)
        print(f"✅ Plan loaded from {plan_file}")
        print(f"   Total expected chunks: {loaded_plan.get('total_expected_chunks', 0):,}")
        print(f"   Number of baskets: {len(loaded_plan.get('baskets', []))}")
        print(f"\n   You can now execute with different percentages:")
        print(f"   results = execute_search(loaded_plan, chunk_percentage=0.2)")
    except Exception as e:
        print(f"❌ Error loading plan: {e}")
else:
    print(f"ℹ️  Plan file '{plan_file}' not found. Save a plan first.")

2026-01-30 16:44:34,238 - INFO - Plan loaded from test_search_plan.json
✅ Plan loaded from test_search_plan.json
   Total expected chunks: 273,029
   Number of baskets: 319

   You can now execute with different percentages:
   results = execute_search(loaded_plan, chunk_percentage=0.2)
